# Blind Spot — Reference Solution

The task: for each `(image, target_class)` row, choose an ordered pair of image
operations that pushes a **hidden** classifier's probability toward the target
class. We never see the hidden model — only its recorded responses on the
training rows.

The plan, in four submissions of increasing strength:

| # | Approach | What it tests |
|---|---|---|
| 1 | Single best constant pair | Is the pipeline wired up correctly? |
| 2 | `(true_class, target_class)` lookup | How far does class information alone go? |
| 3 | CNN surrogate of the response surface | Does image content add signal? |
| 4 | 3-seed ensemble of the surrogate | Does averaging stabilise the argmax? |

The scored quantity is row-normalised, so 1.0 means picking the single best of
the 400 pairs available for that row and 0.0 the worst.

In [1]:
import time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from pathlib import Path

T_START = time.time()
PUB = Path("./dataset/public")
OUT = Path("./working"); OUT.mkdir(parents=True, exist_ok=True)
N_OPS, N_PAIR, SEED = 20, 400, 0

torch.manual_seed(SEED); np.random.seed(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)
if dev.type == "cuda":
    print(torch.cuda.get_device_name(0))

device: cpu


## 1. Load the data

`train_response.csv` is the whole point of the training split: for every
training row it records what the hidden model actually did under all 400 ordered
operation pairs. That is our only window onto its behaviour.

In [2]:
train = pd.read_csv(PUB / "train.csv")
test  = pd.read_csv(PUB / "test.csv")
ops   = pd.read_csv(PUB / "ops.csv")

RCOLS = [f"r{k}" for k in range(N_PAIR)]
resp  = pd.read_csv(PUB / "train_response.csv").set_index("row_id")
resp  = resp.loc[train["row_id"]][RCOLS]
R     = resp.to_numpy(np.float32)

print("train rows", len(train), " test rows", len(test), " response", R.shape)
print("train images", train.image_id.nunique(), " test images", test.image_id.nunique())
print(ops.to_string(index=False))

train rows 10430  test rows 7045  response (10430, 400)
train images 3386  test images 2259
 op_index     op_name  strength_param
        0    identity          0.0000
        1        blur          0.4536
        2      downup          0.4663
        3        jpeg          0.3481
        4       noise          0.3364
        5     speckle          0.0737
        6   bright_up          0.4712
        7   bright_dn          0.5298
        8 contrast_up          0.6558
        9 contrast_dn          0.6411
       10       desat          0.6343
       11      sat_up          0.4302
       12        warm          0.3257
       13        cool          0.3931
       14       gamma          0.2847
       15     sharpen          0.2241
       16    motion_h          0.2505
       17    motion_v          0.2241
       18        zoom          0.5386
       19   posterize          0.7603


The metric normalises each row between its best and worst attainable pair, so we
train against that normalised surface rather than raw probabilities — otherwise
rows with easily-reached targets would dominate the loss.

In [3]:
lo, hi = R.min(1, keepdims=True), R.max(1, keepdims=True)
Rn = (R - lo) / np.maximum(hi - lo, 1e-9)      # row-normalised: what the metric rewards

print(f"attainable p_target: mean max {hi.mean():.3f}  mean min {lo.mean():.3f}")
print(f"rows per image: {len(train) / train.image_id.nunique():.2f}")

attainable p_target: mean max 0.692  mean min 0.001
rows per image: 3.08


## 2. Validation split — grouped by image

An image contributes up to nine rows (one per non-true target class). Splitting
rows at random would put the same image on both sides of the split and inflate
the validation score. We split by `image_id`.

In [4]:
uimg = np.array(sorted(train.image_id.unique()))
rs   = np.random.RandomState(SEED)
val_imgs = set(uimg[rs.permutation(len(uimg))[:int(0.2 * len(uimg))]])
is_val   = train.image_id.isin(val_imgs).to_numpy()
tr_idx, va_idx = np.where(~is_val)[0], np.where(is_val)[0]

def val_score(choice):
    """choice: flat pair index per validation row -> mean row-normalised score"""
    return float(Rn[va_idx, choice].mean())

print(f"fit on {len(tr_idx)} rows / {len(uimg) - len(val_imgs)} images")
print(f"validate on {len(va_idx)} rows / {len(val_imgs)} images")

fit on 8406 rows / 2709 images
validate on 2024 rows / 677 images


## 3. Submission 1 — best constant pair

The simplest thing that can possibly work: find the pair with the highest mean
normalised response across training rows and submit it everywhere. This is
mainly a wiring check, and it establishes the floor.

In [5]:
best_const = Rn[tr_idx].mean(0).argmax()
s1 = val_score(np.full(len(va_idx), best_const))
print(f"best constant pair = ({best_const // N_OPS} -> {best_const % N_OPS})"
      f"  [{ops.op_name[best_const // N_OPS]} -> {ops.op_name[best_const % N_OPS]}]")
print(f"validation score: {s1:.4f}")

sub = pd.DataFrame({"row_id": test.row_id,
                    "op1": best_const // N_OPS, "op2": best_const % N_OPS})
sub.to_csv(OUT / "submission.csv", index=False)
print(sub.head())

best constant pair = (9 -> 9)  [contrast_dn -> contrast_dn]
validation score: 0.1754
      row_id  op1  op2
0  te_000000    9    9
1  te_000001    9    9
2  te_000002    9    9
3  te_000003    9    9
4  te_000004    9    9


## 4. Submission 2 — (class, target) lookup

The response surface has real class structure: what steers a *Forest* patch
toward *Highway* is not what steers *Residential* toward *SeaLake*. Averaging
the normalised response within each `(true_class, target_class)` group and
taking the argmax gives a strong, very cheap baseline.

This is the number a serious solution has to beat.

In [6]:
tr_y, tr_t = train.true_class.to_numpy(), train.target_class.to_numpy()

lookup = {}
for y in range(10):
    for t in range(10):
        m = np.zeros(len(train), bool); m[tr_idx] = True
        m &= (tr_y == y) & (tr_t == t)
        if m.sum() >= 3:
            lookup[(y, t)] = Rn[m].mean(0).argmax()

pick = np.array([lookup.get((y, t), best_const)
                 for y, t in zip(tr_y[va_idx], tr_t[va_idx])])
s2 = val_score(pick)
print(f"validation score: {s2:.4f}   (constant was {s1:.4f})")

pick_test = np.array([lookup.get((y, t), best_const)
                      for y, t in zip(test.true_class, test.target_class)])
sub = pd.DataFrame({"row_id": test.row_id,
                    "op1": pick_test // N_OPS, "op2": pick_test % N_OPS})
sub.to_csv(OUT / "submission.csv", index=False)
print(f"agreement with per-row oracle: "
      f"{(pick == Rn[va_idx].argmax(1)).mean():.1%} of validation rows")

validation score: 0.7024   (constant was 0.1754)
agreement with per-row oracle: 29.5% of validation rows


## 5. Submission 3 — a CNN surrogate of the response surface

The lookup ignores the image. To go further we need a model of *this specific
hidden classifier's* geometry around *this specific image*.

We train a small CNN that takes the image plus embeddings of the true and target
class, and predicts the full 400-length response vector. At inference we take
the argmax. Two things matter:

- **Predict the surface, not the argmax.** Many rows have several near-tied
  pairs; a 400-way classifier on the argmax label throws that structure away.
- **Match the ranking, not just the values.** The loss combines MSE on the
  normalised surface with a soft-listwise KL term, since only the position of
  the maximum is scored.

Training from scratch is deliberate — no pretrained weights are downloaded, so
the notebook is self-contained.

In [7]:
MEAN = np.array([0.3444, 0.3803, 0.4078], np.float32)
STD  = np.array([0.2027, 0.1369, 0.1156], np.float32)

_cache = {}
def img_array(i):
    if i not in _cache:
        _cache[i] = np.asarray(Image.open(PUB / "images" / f"{i}.png").convert("RGB"))
    return _cache[i]

def make_x(ids):
    a = np.stack([img_array(i) for i in ids]).astype(np.float32) / 255.
    a = (a - MEAN) / STD
    return torch.from_numpy(a.transpose(0, 3, 1, 2).copy())

Xtr = make_x(train.image_id.tolist())
Xte = make_x(test.image_id.tolist())
Ytr, Ttr = torch.from_numpy(tr_y), torch.from_numpy(tr_t)
Yte = torch.from_numpy(test.true_class.to_numpy())
Tte = torch.from_numpy(test.target_class.to_numpy())
Rt  = torch.from_numpy(Rn)
print("image tensors:", tuple(Xtr.shape), tuple(Xte.shape))

image tensors: (10430, 3, 64, 64) (7045, 3, 64, 64)


/var/folders/gd/wj93f8jd0dbf4c16x2hf9d8r0000gn/T/ipykernel_20898/3279456940.py:17: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  Ytr, Ttr = torch.from_numpy(tr_y), torch.from_numpy(tr_t)


In [8]:
class Surrogate(nn.Module):
    """(image, true_class, target_class) -> predicted response over all 400 pairs."""
    def __init__(self, d=192):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(2))
        self.f    = nn.Sequential(blk(3, 32), blk(32, 64), blk(64, 128), blk(128, d))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.yemb, self.temb = nn.Embedding(10, 64), nn.Embedding(10, 64)
        self.head = nn.Sequential(
            nn.Linear(d + 128, 512), nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, N_PAIR))

    def forward(self, x, y, t):
        h = self.pool(self.f(x)).flatten(1)
        return self.head(torch.cat([h, self.yemb(y), self.temb(t)], 1))


def predict(net, X, Y, T):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), 512):
            out.append(net(X[i:i+512].to(dev), Y[i:i+512].to(dev),
                           T[i:i+512].to(dev)).float().cpu().numpy())
    return np.concatenate(out)


def train_surrogate(epochs=40, seed=0, verbose=True):
    torch.manual_seed(seed)
    net = Surrogate().to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
    nstep = epochs * int(np.ceil(len(tr_idx) / 256))
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=nstep)
    rs = np.random.RandomState(seed)
    for ep in range(epochs):
        net.train()
        perm = rs.permutation(tr_idx)
        for i in range(0, len(perm), 256):
            b = perm[i:i+256]
            # no flip/rotation augmentation: the recorded response belongs to
            # THIS image, and directional ops (motion_h vs motion_v) make a
            # transformed image's true response a different quantity
            xb = Xtr[b].to(dev)
            pred, tgt = net(xb, Ytr[b].to(dev), Ttr[b].to(dev)), Rt[b].to(dev)
            loss = F.mse_loss(pred, tgt) + 0.5 * F.kl_div(
                F.log_softmax(pred * 3, 1), F.softmax(tgt * 3, 1),
                reduction="batchmean")
            opt.zero_grad(); loss.backward(); opt.step(); sch.step()
        if verbose and (ep + 1) % 10 == 0:
            pv = predict(net, Xtr[va_idx], Ytr[va_idx], Ttr[va_idx])
            print(f"  ep{ep+1:3d}  val {val_score(pv.argmax(1)):.4f}  "
                  f"({time.time()-T_START:.0f}s)", flush=True)
    return net

In [9]:
net = train_surrogate(epochs=40, seed=0)
pv  = predict(net, Xtr[va_idx], Ytr[va_idx], Ttr[va_idx])
s3  = val_score(pv.argmax(1))
print(f"\nsurrogate validation: {s3:.4f}   (lookup was {s2:.4f})")

pt = predict(net, Xte, Yte, Tte).argmax(1)
sub = pd.DataFrame({"row_id": test.row_id, "op1": pt // N_OPS, "op2": pt % N_OPS})
sub.to_csv(OUT / "submission.csv", index=False)

  ep 10  val 0.7304  (594s)


  ep 20  val 0.7942  (1198s)


  ep 30  val 0.8027  (1807s)


  ep 40  val 0.8058  (2395s)



surrogate validation: 0.8058   (lookup was 0.7024)


## 6. Submission 4 — seed ensemble (final)

The argmax over 400 near-tied options is noisy, so averaging predicted surfaces
across independently seeded runs is worth more here than it would be on a
typical classification task. We average three runs and take the argmax of the
mean surface.

In [10]:
acc = pv.copy()
acc_test = predict(net, Xte, Yte, Tte)

for s in (1, 2):
    n2 = train_surrogate(epochs=40, seed=s, verbose=False)
    acc      += predict(n2, Xtr[va_idx], Ytr[va_idx], Ttr[va_idx])
    acc_test += predict(n2, Xte, Yte, Tte)
    print(f"  +seed{s}: val {val_score(acc.argmax(1)):.4f}  "
          f"({time.time()-T_START:.0f}s)", flush=True)

s4 = val_score(acc.argmax(1))
final = acc_test.argmax(1)
sub = pd.DataFrame({"row_id": test.row_id,
                    "op1": final // N_OPS, "op2": final % N_OPS})
sub.to_csv(OUT / "submission.csv", index=False)
print(f"\nfinal validation: {s4:.4f}")

  +seed1: val 0.8109  (4880s)


  +seed2: val 0.8141  (7285s)



final validation: 0.8141


## 7. Check the submission and summarise

In [11]:
chk = pd.read_csv(OUT / "submission.csv")
assert list(chk.columns) == ["row_id", "op1", "op2"], chk.columns
assert len(chk) == len(test), (len(chk), len(test))
assert set(chk.row_id) == set(test.row_id)
assert not chk.row_id.duplicated().any()
assert chk.op1.between(0, 19).all() and chk.op2.between(0, 19).all()
assert chk.dtypes["op1"].kind == "i" and chk.dtypes["op2"].kind == "i"
print(f"submission OK: {len(chk)} rows")

print(f"\n{'approach':<34}{'val score'}")
print(f"{'-'*46}")
for nm, v in [("1. best constant pair", s1),
              ("2. (class,target) lookup", s2),
              ("3. CNN surrogate", s3),
              ("4. 3-seed ensemble  <- submitted", s4)]:
    print(f"{nm:<34}{v:.4f}")
print(f"\ntotal runtime: {(time.time()-T_START)/60:.1f} min")
print(f"unique pairs used: {len(set(zip(chk.op1, chk.op2)))} of 400")

submission OK: 7045 rows

approach                          val score
----------------------------------------------
1. best constant pair             0.1754
2. (class,target) lookup          0.7024
3. CNN surrogate                  0.8058
4. 3-seed ensemble  <- submitted  0.8141

total runtime: 121.4 min
unique pairs used: 197 of 400
